In [33]:
import os
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.layers import InputLayer,Conv2D,MaxPooling2D,Flatten,Dense
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.layers import Layer
from keras.layers import Reshape
from keras.layers import Lambda
from keras import initializers


In [2]:
def preprocessing(path):
    img = Image.open(path)
    img_resized = img.resize((224,224))
    return np.array(img_resized)/255

In [3]:
data_path = '../data/spectrograms'
class0 = 'class_0'
class1 = 'class_1'

In [4]:
non_seizure = []
seizure = []
n_seizure_names = [f for f in os.listdir(os.path.join(data_path,class0))]
seizure_names = [f for f in os.listdir(os.path.join(data_path,class1))]
for path in n_seizure_names:
    non_seizure.append(preprocessing(os.path.join(data_path,class0,path)))
for path in seizure_names:
    seizure.append(preprocessing(os.path.join(data_path,class1,path)))

In [5]:
non_seizure_labels = [0]*len(non_seizure)
seizure_labels = [1]*len(seizure)

data = non_seizure+seizure
data_labels = non_seizure_labels+seizure_labels

X, y = shuffle(np.array(data), np.eye(2)[data_labels],random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,random_state = 42)
X_test, X_val, y_test, y_val = train_test_split(X_test,y_test,test_size=1/3,random_state = 42)

In [8]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    '../weights/CapsuleNET.weights.h5',
    monitor='val_loss',
    save_best_only=True,
    save_weights_only=True
)

In [30]:
def squash(inputs):
    squared_norm = K.sum(K.square(inputs), axis=-1, keepdims=True)
    return (squared_norm / (1 + squared_norm)) / (K.sqrt(squared_norm + K.epsilon())) * inputs

class DigitCapsuleLayer(Layer):
    def __init__(self, **kwargs):
        super(DigitCapsuleLayer, self).__init__(**kwargs)
        self.kernel_initializer = initializers.get('glorot_uniform')

    def build(self, input_shape):
        self.W = self.add_weight(
            shape=[2, input_shape[1], 16, 8],  # 2 capsules for binary
            initializer=self.kernel_initializer,
            name='weights'
        )
        self.built = True

    def call(self, inputs):
        inputs_expanded = K.expand_dims(inputs, 1)
        inputs_tiled = K.tile(inputs_expanded, [1, 2, 1, 1])
        u_hat = K.map_fn(lambda x: K.batch_dot(x, self.W, [2, 3]), elems=inputs_tiled)
        b = tf.zeros(shape=[K.shape(inputs)[0], 2, input_shape[1]])

        for i in range(3 - 1):
            c = tf.nn.softmax(b, axis=1)
            s = K.batch_dot(c, u_hat, [2, 2])
            v = squash(s)
            b = b + K.batch_dot(v, u_hat, [2, 3])

        return v

    def compute_output_shape(self, input_shape):
        return (None, 2, 16)

def output_layer(inputs):
    return K.sqrt(K.sum(K.square(inputs), -1) + K.epsilon())

def mask(outputs):
    if type(outputs) != list:
        norm_outputs = K.sqrt(K.sum(K.square(outputs), -1) + K.epsilon())
        y = K.one_hot(indices=K.argmax(norm_outputs, 1), num_classes=2)
        y = Reshape((2, 1))(y)
        return Flatten()(y * outputs)
    else:
        y = Reshape((2, 1))(outputs[1])
        masked_output = y * outputs[0]
        return Flatten()(masked_output)

def loss_fn(y_true, y_pred):
    L = y_true * K.square(K.maximum(0., 0.9 - y_pred)) + \
        0.5 * (1 - y_true) * K.square(K.maximum(0., y_pred - 0.1))
    return K.mean(K.sum(L, 1))

In [34]:
input_shape = Input(shape=(224, 224, 4))  # updated input

# conv layers
conv1 = Conv2D(256, (9, 9), activation='relu', padding='valid')(input_shape)
conv2 = Conv2D(256, (9, 9), strides=2, padding='valid')(conv1)

# reshape
reshaped = Reshape((conv2.shape[1]*conv2.shape[2]*32, 8))(conv2)

squashed_output = Lambda(squash, output_shape=lambda s: s)(reshaped)

digit_caps = DigitCapsuleLayer()(squashed_output)

outputs = Lambda(output_layer)(digit_caps)

inputs_label = Input(shape=(2,))
masked = Lambda(mask)([digit_caps, inputs_label])
masked_for_test = Lambda(mask)(digit_caps)

# decoder
decoded_inputs = Input(shape=(16 * 2,))
dense1 = Dense(512, activation='relu')(decoded_inputs)
dense2 = Dense(1024, activation='relu')(dense1)
decoded_outputs = Dense(224 * 224 * 4, activation='sigmoid')(dense2)
decoded_outputs = Reshape((224, 224, 4))(decoded_outputs)

decoder = Model(decoded_inputs, decoded_outputs)
model = Model([input_shape, inputs_label], [outputs, decoder(masked)])
test_model = Model(input_shape, [outputs, decoder(masked_for_test)])

2025-05-28 08:29:09.335993: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:497] Allocator (GPU_0_bfc) ran out of memory trying to allocate 338.00MiB (rounded to 354418688)requested by op AddV2
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2025-05-28 08:29:09.336387: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1053] BFCAllocator dump for GPU_0_bfc
2025-05-28 08:29:09.336484: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1060] Bin (256): 	Total Chunks: 38, Chunks in use: 38. 9.5KiB allocated for chunks. 9.5KiB in use in bin. 716B client-requested in use in bin.
2025-05-28 08:29:09.336491: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1060] Bin (512): 	Total Chunks: 2, Chunks in use: 2. 1.0KiB allocated for chunks. 1.0KiB in use in bin. 1.0KiB client-requested in use in bin.
2025-05-28 08:29

ResourceExhaustedError: {{function_node __wrapped__AddV2_device_/job:localhost/replica:0/task:0/device:GPU:0}} failed to allocate memory [Op:AddV2] name: 

605700 of size 1536 next 53
2025-05-28 08:29:09.336753: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1109] InUse at 505605d00 of size 1024 next 55
2025-05-28 08:29:09.336755: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1109] InUse at 505606100 of size 1024 next 57
2025-05-28 08:29:09.336758: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1109] InUse at 505606500 of size 112128 next 25
2025-05-28 08:29:09.336760: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1109] InUse at 505621b00 of size 65536 next 26
2025-05-28 08:29:09.336763: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1109] InUse at 505631b00 of size 463104 next 10
2025-05-28 08:29:09.336766: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1109] InUse at 5056a2c00 of size 331776 next 11
2025-05-28 08:29:09.336768: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1109] InUse at 5056f3c00 of size 802816 next 27
2025-05-28 08:29:09.336771: I external/local_xla/xla/

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.000001),
              loss=[loss_fn, 'mse'],
              loss_weights=[1., 0.0005],
              metrics=['accuracy'])

In [ ]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)
checkpoint = ModelCheckpoint(
    'weights/RNN_LSTM.weights.h5',
    monitor='val_loss',
    save_best_only=True,
    save_weights_only=True
)

In [ ]:
history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=30,
        batch_size = 20,
        callbacks= [early_stopping,checkpoint]
    )